# Multi-Entity Interference in Retrieval-Augmented Language Models

**Reproducible End-to-End Evaluation & Statistical Verification Notebook (`Qwen2.5-7B-Instruct`)**  

- **Model:** `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4 Quantized, Greedy Decoding $\tau = 0.0$)
- **Experimental Design:** $2 \times 2$ Full Factorial Matrix ($N = 200$ scenarios across GeoAI Spatial Infrastructure and Clinical Trial Protocols)
- **Audit Protocol:** Deterministic Zero-LLM Bipartite Matching Engine


In [ ]:
# [1] Environment Setup (Auto-clones repository when executed in Google Colab)
import os
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
    REPO_URL = "https://github.com/sayem088/Multi_Entity_Interference_in_Retrieval-Augmented_LLMs.git"
    if not os.path.exists("Multi_Entity_Interference_in_Retrieval-Augmented_LLMs"):
        !git clone -q {REPO_URL}
    %cd Multi_Entity_Interference_in_Retrieval-Augmented_LLMs
    !pip install -q -e .
except ImportError:
    IN_COLAB = False
    if Path.cwd().name == "notebooks":
        os.chdir("..")

print(f"Working Directory: {Path.cwd()}")


In [ ]:
# [2] Load Verified 200-Scenario Evaluation Dataset & Run Factorial ANOVA
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy import stats

df = pd.read_csv("experiments/qwen2_5_7b/summary_qwen_7b.csv")
df["Factor_Sim"] = df["cell_label"].apply(lambda s: "Low" if "low_sim" in str(s).lower() else "High")
df["Factor_Cnt"] = df.apply(lambda r: "Low (k=2)" if ("low_count" in str(r["cell_label"]).lower() or r["entity_count"] == 2) else "High", axis=1)

print(f"Loaded N = {len(df)} audited scenarios across {df['domain_id'].nunique()} domains.")
display(df.groupby("cell_label")[["EA_precision", "EA_recall", "EA_F1", "RAGAS_faithfulness", "blindness_gap"]].mean().round(4))

ols_gap = ols("blindness_gap ~ C(Factor_Sim) + C(Factor_Cnt) + C(Factor_Sim):C(Factor_Cnt)", data=df).fit()
anova_gap = sm.stats.anova_lm(ols_gap, typ=2)
ss_res = anova_gap.loc["Residual", "sum_sq"]
anova_gap["eta_p_sq"] = anova_gap["sum_sq"] / (anova_gap["sum_sq"] + ss_res)
print("\nTwo-Way Factorial ANOVA on Evaluation Blindness Gap (Delta_blind):")
display(anova_gap.round(5))

t_stat, p_two = stats.ttest_1samp(df["blindness_gap"].dropna(), 0.0)
print(f"Paired t-test (H_a: Delta_blind > 0): t = {t_stat:.4f}, p = {p_two/2:.4e}")


In [ ]:
# [3] Generate Publication Figures Directly from CSV
import matplotlib.pyplot as plt
import numpy as np

order = ["low_sim_low_count", "high_sim_low_count", "low_sim_high_count", "high_sim_high_count"]
labels = ["Cell 1\nDisjoint Low", "Cell 2\nIsomorphic Low", "Cell 3\nDisjoint High", "Cell 4\nIsomorphic High"]
stats_df = df.groupby("cell_label")[["RAGAS_faithfulness", "EA_recall", "EA_F1", "blindness_gap"]].mean().reindex(order)

fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=150)
x = np.arange(len(order))
w = 0.34
ax.bar(x - w/2, stats_df["RAGAS_faithfulness"], w, label="Standard RAGAS Faithfulness", color="#059669")
ax.bar(x + w/2, stats_df["EA_recall"], w, label="True Entity-Attributed Recall", color="#ea580c")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score (0.0 to 1.0)", fontweight="bold")
ax.set_title("Evaluation Blindness Gap Across Factorial Conditions (Qwen2.5-7B)", fontweight="bold")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
